<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/educacao/IDEB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Ideb — as três etapas da educação básica

Este notebook é o motor de coleta e publicação das séries `ideb-anos-iniciais`,
`ideb-anos-finais` e `ideb-ensino-medio`.

- **Fonte:** Inep — Índice de Desenvolvimento da Educação Básica (Ideb), planilha
  **Brasil** da edição vigente (`divulgacao_brasil_ideb_AAAA.zip`), abas
  **Brasil (Anos Iniciais)**, **Brasil (Anos Finais)** e **Brasil (EM)**, rede **Total**
  (pública + privada).
- **Definição:** índice de 0 a 10 que combina o desempenho dos estudantes no Saeb com a
  taxa de aprovação apurada pelo Censo Escolar, para cada etapa: anos iniciais (1º ao 5º
  ano) e anos finais (6º ao 9º ano) do ensino fundamental regular e ensino médio regular.
- **Fórmula oficial:** `Ideb = N × P`, em que `N` é a nota média padronizada do Saeb
  (Língua Portuguesa e Matemática, 0–10) e `P` é o indicador de rendimento (taxa de
  aprovação, 0–1). O notebook **não recalcula** o índice: lê a coluna
  `VL_OBSERVADO_AAAA` já calculada pelo Inep, que é a fonte oficial.
- **Unidade:** índice (`unidade.tipo: numero`, `simbolo: índice`, `casas: 1`);
  `eixo: contagem`, `medida: nivel`, `granularidade: bienal`.
- **Periodicidade:** o Saeb é aplicado a cada dois anos e o Ideb só existe nos anos
  ímpares, desde a primeira edição (2005). Não há ponto em ano par nem interpolação.
- **Período publicado:** da primeira edição (2005) até a última edição divulgada; o
  notebook grava todos os anos ímpares fechados.
- **Arquivos gerados:** `indicadores/educacao/ideb-anos-iniciais.json`,
  `indicadores/educacao/ideb-anos-finais.json` e
  `indicadores/educacao/ideb-ensino-medio.json`.

## O que este número é — e o que ele não é

- **É o Ideb observado da rede total do Brasil**, públicas e privadas somadas, como o Inep
  publica. Não é a rede pública isolada, que tem Ideb menor em todas as etapas, nem a
  privada.
- **Não é percentual de aprendizado** nem nota de prova: é um índice composto, numa escala
  de 0 a 10, que mistura proficiência (Saeb) e fluxo (aprovação). Um Ideb 6 não quer
  dizer "60% de aprendizado".
- **Não é a nota de uma escola, município ou estado.** É o agregado nacional de cada etapa;
  os recortes subnacionais têm arquivos próprios no Inep.
- **As três etapas não são comparáveis como se fossem a mesma prova.** Os anos iniciais, os
  anos finais e o ensino médio têm provas e metas próprias; a distância entre elas é parte
  do diagnóstico, não uma escala única.
- **Não é a meta do Ideb.** A meta é uma projeção pactuada, não um valor observado; a
  coluna de metas da planilha (`VL_PROJECAO_AAAA`) fica de fora.
- **A pandemia quebra a leitura de tendência.** A passagem de 2019 para 2021 não é a de um
  biênio normal: nos anos iniciais o Ideb caiu (5,9 → 5,8), no ensino médio ficou estável
  (4,2 → 4,2) e nos anos finais subiu (4,9 → 5,1) porque a taxa de aprovação aumentou, não
  porque a aprendizagem tenha melhorado. O Inep publicou nota informativa sobre o impacto da
  pandemia no Ideb de 2021.

## Proveniência e corte do ano corrente

A URL do zip muda a cada edição (o nome carrega o ano). O notebook lê a página de resultados
do Inep, descobre a edição vigente e confere o md5 do XLSX; se o link não estiver lá, falha
em vez de publicar a safra antiga. O Ideb de um ano só é divulgado no ano seguinte, então o
ano corrente nunca entra.

## Setup e raiz

`DADOS_RAIZ` permite rodar contra uma cópia do repositório e `EDUCACAO_SAIDA` grava os JSONs
num diretório de ensaio. Todo caminho exibido passa por `relativo()`: as saídas do notebook
são públicas e não podem carregar o diretório da máquina.

In [1]:
import hashlib
import io
import json
import math
import os
import re
import ssl
import tempfile
import time
import xml.etree.ElementTree as ET
import zipfile
from datetime import date
from pathlib import Path
from urllib.parse import urljoin

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "educacao"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("EDUCACAO_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


# Página estável de resultados do Ideb: as abas (2005|2025, 2005|2027, ...) apontam para a
# edição vigente. É também o urlFonte da série.
PAGINA_RESULTADOS = (
    "https://www.gov.br/inep/pt-br/areas-de-atuacao/pesquisas-estatisticas-e-indicadores/"
    "ideb/resultados"
)
PADRAO_BRASIL = re.compile(r"divulgacao_brasil_ideb_(?P<edicao>\d{4})\.zip$")
HOST_DOWNLOAD = "https://download.inep.gov.br/"
# Endereço AIA do certificado de download.inep.gov.br, que não envia o intermediário da
# cadeia. Ver a explicação em `preparar_sessao()`.
CA_INTERMEDIARIA = "https://secure.globalsign.com/cacert/rnpicpedugr46ovtlsca2025.crt"
PRIMEIRO_ANO = 2005
# O Ideb de um ano só é divulgado no ano seguinte: o ano corrente nunca está fechado.
ANO_CORTE = date.today().year - 1
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
RID = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"


## Coleta: descoberta da edição e leitura do XLSX

A página de resultados do Inep é lida primeiro para localizar a aba vigente
(`data-url=".../resultados/2005-AAAA"`); a aba traz o link
`divulgacao_brasil_ideb_AAAA.zip`. O notebook fica com a maior edição disponível e falha
se o zip não estiver listado — nunca cai para a edição anterior.

O zip traz um `.xlsx` e um `.ods` com o mesmo conteúdo e um `.txt` com o md5. O notebook
usa o `.xlsx` (o `.ods` é outro formato do mesmo dado) e, quando o md5 está presente,
confere a integridade do arquivo. A leitura é feita com `zipfile` + XML da biblioteca
padrão, como no notebook da População: não é preciso instalar `openpyxl` para ler uma
tabela estática de 33 números.

### TLS do `download.inep.gov.br`

O servidor de downloads do Inep entrega **só o certificado do site**, sem o intermediário
da cadeia (RNP ICPEdu), e o bundle de raízes do `requests` não o tem: a verificação falha
com `unable to get local issuer certificate`. O notebook tenta a conexão normal e, se ela
falhar assim, baixa o intermediário público no endereço AIA do próprio certificado e refaz
a sessão com **raízes + intermediário**. A verificação continua ligada de ponta a ponta — o
intermediário é validado contra a raiz GlobalSign na mesma conexão. Se o endereço mudar, o
download falha; o TLS nunca é desligado.

In [2]:
def preparar_sessao():
    """Sessão do Inep com a cadeia TLS completa.

    `download.inep.gov.br` serve só o certificado do site, sem o intermediário; o bundle
    padrão do requests só traz raízes, e a verificação falha com "unable to get local
    issuer certificate". Quando isso acontece, o notebook baixa o intermediário público no
    endereço AIA do próprio certificado e refaz a sessão com raízes + intermediário. A
    verificação segue ligada: o intermediário é validado contra a raiz GlobalSign na mesma
    conexão. Se o endereço mudar, o download falha — o TLS nunca é desligado.
    """
    sessao = requests.Session()
    try:
        sessao.head(HOST_DOWNLOAD, timeout=60).raise_for_status()
        return sessao
    except requests.RequestException:
        pass

    resposta = requests.get(CA_INTERMEDIARIA, timeout=60)
    resposta.raise_for_status()
    intermediaria = resposta.content
    if not intermediaria.lstrip().startswith(b"-----BEGIN CERTIFICATE-----"):
        intermediaria = ssl.DER_cert_to_PEM_cert(intermediaria).encode()
    raizes = Path(requests.certs.where()).read_bytes()
    if not raizes.endswith(b"\n"):
        raizes += b"\n"
    bundle = Path(tempfile.mkdtemp()) / "inep-cadeia.pem"
    bundle.write_bytes(raizes + intermediaria)
    sessao.verify = str(bundle)
    return sessao


def baixar_texto(sessao, url, tentativas=4, timeout=120):
    """GET de página HTML com repetição; o gov.br falha de forma intermitente."""
    for tentativa in range(tentativas):
        try:
            resposta = sessao.get(url, timeout=timeout)
            resposta.raise_for_status()
            return resposta.text
        except requests.RequestException:
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def baixar_zip(sessao, url, tentativas=4, timeout=180):
    """GET do .zip com repetição e conferência de integridade."""
    for tentativa in range(tentativas):
        try:
            resposta = sessao.get(url, timeout=timeout)
            resposta.raise_for_status()
            if not resposta.content.startswith(b"PK"):
                raise ValueError(f"{url} não devolveu um .zip")
            with zipfile.ZipFile(io.BytesIO(resposta.content)) as pacote:
                if pacote.testzip() is not None:
                    raise ValueError(f"{url}: arquivo corrompido")
            return resposta.content
        except (requests.RequestException, ValueError, zipfile.BadZipFile):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def edicoes_disponiveis(sessao):
    """Mapa edição -> URL do zip Brasil, lido nas abas da página de resultados.

    A aba vigente aponta para a edição mais nova; as anteriores saem do ar. Ler a página
    (em vez de fixar a URL) é o que impede o notebook de publicar a safra antiga quando o
    Inep troca o arquivo.
    """
    indice = baixar_texto(sessao, PAGINA_RESULTADOS)
    abas = {
        urljoin(PAGINA_RESULTADOS, url)
        for url in re.findall(r'data-url="([^"]+)"', indice)
        if "/resultados/" in url
    }
    abas.add(PAGINA_RESULTADOS)
    arquivos = {}
    for aba in sorted(abas):
        html = indice if aba == PAGINA_RESULTADOS else baixar_texto(sessao, aba)
        for link in re.findall(r'href="([^"]+)"', html):
            url = urljoin(aba, link)
            casamento = PADRAO_BRASIL.search(url)
            if casamento and int(casamento["edicao"]) <= date.today().year:
                arquivos[int(casamento["edicao"])] = url
    return arquivos


def folhas(xlsx):
    """Mapa nome da aba -> caminho do XML dentro do .xlsx."""
    with zipfile.ZipFile(io.BytesIO(xlsx)) as pacote:
        livro = ET.fromstring(pacote.read("xl/workbook.xml"))
        relacoes = ET.fromstring(pacote.read("xl/_rels/workbook.xml.rels"))
        alvos = {relacao.get("Id"): relacao.get("Target") for relacao in relacoes}
        return {
            folha.get("name"): "xl/" + alvos[folha.get(RID)].lstrip("/").replace("xl/", "")
            for folha in livro.iter(NS + "sheet")
        }


def linhas(xlsx, caminho):
    """Linhas da aba como dicionários coluna -> texto (zip de XML, sem openpyxl)."""
    with zipfile.ZipFile(io.BytesIO(xlsx)) as pacote:
        compartilhadas = []
        if "xl/sharedStrings.xml" in pacote.namelist():
            for item in ET.fromstring(pacote.read("xl/sharedStrings.xml")):
                compartilhadas.append("".join(t.text or "" for t in item.iter(NS + "t")))
        folha = ET.fromstring(pacote.read(caminho))
        saida = []
        for linha in folha.iter(NS + "row"):
            valores = {}
            for celula in linha.iter(NS + "c"):
                coluna = "".join(ch for ch in celula.get("r") if ch.isalpha())
                bruto = celula.find(NS + "v")
                if bruto is None:
                    embutido = celula.find(NS + "is")
                    valor = (
                        "".join(t.text or "" for t in embutido.iter(NS + "t"))
                        if embutido is not None
                        else None
                    )
                elif celula.get("t") == "s":
                    valor = compartilhadas[int(bruto.text)]
                else:
                    valor = bruto.text
                valores[coluna] = valor
            saida.append(valores)
        return saida


sessao = preparar_sessao()
edicoes = edicoes_disponiveis(sessao)
assert edicoes, f"nenhum zip Brasil do Ideb listado em {PAGINA_RESULTADOS}"
EDICAO = max(edicoes)
URL_ARQUIVO = edicoes[EDICAO]
print(f"edição disponível: {EDICAO} ({URL_ARQUIVO.rsplit('/', 1)[-1]})")

conteudo = baixar_zip(sessao, URL_ARQUIVO)
with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
    nomes = pacote.namelist()
    nome_xlsx = next(
        (nome for nome in nomes if re.search(rf"/divulgacao_brasil_ideb_{EDICAO}\.xlsx$", nome)),
        None,
    )
    assert nome_xlsx, f"planilha Brasil do Ideb {EDICAO} não está no zip: {nomes}"
    xlsx = pacote.read(nome_xlsx)
    nome_md5 = next((nome for nome in nomes if nome.endswith(".txt") and "md5" in nome), None)
    if nome_md5 is not None:
        esperado = hashlib.md5(xlsx).hexdigest()
        texto = pacote.read(nome_md5).decode("utf-8", "replace")
        casamento = re.search(
            r"([0-9A-Fa-f]{32})\s+\*?" + re.escape(Path(nome_xlsx).name), texto
        )
        assert casamento, f"{nome_md5} não confere {Path(nome_xlsx).name}"
        assert casamento.group(1).lower() == esperado, (casamento.group(1), esperado)
        print(f"md5 confere: {esperado}")

print(f"planilha usada: {Path(nome_xlsx).name} ({len(folhas(xlsx))} abas)")


edição disponível: 2025 (divulgacao_brasil_ideb_2025.zip)


md5 confere: 94c8e530337c86251de705ddae42afc1
planilha usada: divulgacao_brasil_ideb_2025.xlsx (3 abas)


## Seleção do recorte: Brasil, rede total, três etapas

As três abas têm o mesmo desenho: uma linha de cabeçalho visível `IDEB AAAA (N x P)`, uma
linha de códigos `VL_OBSERVADO_AAAA` e cinco linhas de rede (`Total`, `Estadual`,
`Municipal`, `Pública` e `Privada`). O notebook usa a linha `Brasil`/`Total` — a mesma que o
Inep destaca no relatório de resultados.

O ano de cada coluna vem do **cabeçalho visível**, não da linha de códigos: na aba de anos
finais da edição de 2025, a linha de códigos pula 2023, mas a coluna "IDEB 2023 (N x P)"
existe e tem o valor 5,0. Ler pelos códigos perderia o ponto em silêncio. O código, quando
presente na coluna, é conferido contra o ano do cabeçalho — se a planilha desalinhar, o
notebook falha.

O corte é explícito: só entram anos entre `PRIMEIRO_ANO` e `ANO_CORTE`; um ano par na fonte
**não é descartado em silêncio** — ele quebra a premissa de que o Ideb é bienal ímpar e o
notebook falha, porque aí a decisão é de contrato, não de código.

In [3]:
ETAPAS = {
    "ideb-anos-iniciais": "Brasil (Anos Iniciais)",
    "ideb-anos-finais": "Brasil (Anos Finais)",
    "ideb-ensino-medio": "Brasil (EM)",
}
PADRAO_IDEB = re.compile(r"IDEB\s+(\d{4})")


def selecionar(aba):
    """Ideb observado da rede total do Brasil na aba pedida, por ano."""
    nomes = folhas(xlsx)
    assert aba in nomes, f"aba {aba!r} não encontrada: {sorted(nomes)}"
    grade = linhas(xlsx, nomes[aba])

    # Ano pelo cabeçalho visível ('IDEB\n2005\n(N x P)'): a linha VL_OBSERVADO da aba de
    # anos finais pula 2023, embora o valor exista na coluna. Código é conferência, não mapa.
    cabecalho = next(
        (
            linha
            for linha in grade
            if any(PADRAO_IDEB.match((valor or "").strip()) for valor in linha.values())
        ),
        None,
    )
    assert cabecalho, f"cabeçalho do Ideb não encontrado em {aba!r}"
    colunas = {
        int(casamento.group(1)): coluna
        for coluna, valor in cabecalho.items()
        if (casamento := PADRAO_IDEB.match((valor or "").strip()))
    }
    assert colunas, f"colunas IDEB AAAA não encontradas em {aba!r}"

    codigos = next(
        (
            linha
            for linha in grade
            if any(re.fullmatch(r"VL_OBSERVADO_\d{4}", valor or "") for valor in linha.values())
        ),
        {},
    )
    for ano, coluna in colunas.items():
        codigo = (codigos.get(coluna) or "").strip()
        if codigo:
            assert codigo == f"VL_OBSERVADO_{ano}", (aba, ano, codigo)

    # Rede total do Brasil — não é 'Pública', 'Privada' nem outra etapa de ensino.
    candidatas = [
        linha
        for linha in grade
        if (linha.get("A") or "").strip() == "Brasil"
        and (linha.get("B") or "").strip() == "Total"
    ]
    assert len(candidatas) == 1, f"{aba!r}: linha Brasil/Total com {len(candidatas)} ocorrências"
    linha_total = candidatas[0]

    observado = {
        ano: float(linha_total[coluna])
        for ano, coluna in sorted(colunas.items())
        if linha_total.get(coluna) not in (None, "")
    }

    # Corte do ano corrente: o Ideb de um ano só fecha no ano seguinte. A série é bienal
    # ímpar; ano par aqui é mudança de metodologia, não ponto a descartar em silêncio.
    anos = sorted(ano for ano in observado if PRIMEIRO_ANO <= ano <= ANO_CORTE)
    assert anos, f"nenhum ano do Ideb entre {PRIMEIRO_ANO} e {ANO_CORTE} em {aba!r}"
    pares = [ano for ano in anos if ano % 2 == 0]
    assert not pares, (
        f"anos pares no Ideb ({pares}, {aba!r}): a série é bienal ímpar. Se a fonte mudou "
        "de periodicidade, o contrato e a granularidade têm de ser revistos antes de publicar."
    )
    assert anos[0] == PRIMEIRO_ANO, (aba, anos[0], PRIMEIRO_ANO)
    assert anos[-1] == EDICAO, (aba, anos[-1], EDICAO)
    assert anos == list(range(PRIMEIRO_ANO, EDICAO + 1, 2)), (aba, anos)
    return {ano: observado[ano] for ano in anos}


ideb = {id_serie: selecionar(aba) for id_serie, aba in ETAPAS.items()}
for id_serie, valores in ideb.items():
    print(f"\n{id_serie} ({len(valores)} pontos)")
    for ano, valor in valores.items():
        print(f"  {ano}: {valor:.1f}")


ideb-anos-iniciais (11 pontos)
  2005: 3.8
  2007: 4.2
  2009: 4.6
  2011: 5.0
  2013: 5.2
  2015: 5.5
  2017: 5.8
  2019: 5.9
  2021: 5.8
  2023: 6.0
  2025: 6.3

ideb-anos-finais (11 pontos)
  2005: 3.5
  2007: 3.8
  2009: 4.0
  2011: 4.1
  2013: 4.2
  2015: 4.5
  2017: 4.7
  2019: 4.9
  2021: 5.1
  2023: 5.0
  2025: 5.3

ideb-ensino-medio (11 pontos)
  2005: 3.4
  2007: 3.5
  2009: 3.6
  2011: 3.7
  2013: 3.7
  2015: 3.7
  2017: 3.8
  2019: 4.2
  2021: 4.2
  2023: 4.3
  2025: 4.5


## Escrita dos JSONs do contrato

Cada arquivo é gravado inteiro (nada de merge com o que está publicado), com `versao`, `id`,
`unidade`, `eixo`, `medida`, `granularidade`, `fonte`, `urlFonte` e `dados` no formato
`{"data": "YYYY-MM-DD", "valor": ...}`. O valor vai **sem arredondamento**; `unidade.casas`
é só exibição.

`urlFonte` é a página de resultados do Ideb do Inep — o endereço estável que lista a edição
vigente e de onde o notebook descobre o zip. O arquivo em si muda a cada edição; o nome do
zip e o da planilha usados ficam registrados na saída da coleta.

In [4]:
UNIDADE = {"tipo": "numero", "simbolo": "índice", "casas": 1}
NOMES = {
    "ideb-anos-iniciais": "anos iniciais do ensino fundamental",
    "ideb-anos-finais": "anos finais do ensino fundamental",
    "ideb-ensino-medio": "ensino médio",
}

series = {}
for id_serie, valores in ideb.items():
    series[id_serie] = {
        "versao": 1,
        "id": id_serie,
        "unidade": UNIDADE,
        "eixo": "contagem",
        "medida": "nivel",
        "granularidade": "bienal",
        "fonte": f"Inep — Ideb, {NOMES[id_serie]}, rede total (Brasil)",
        "urlFonte": PAGINA_RESULTADOS,
        "dados": [{"data": f"{ano}-01-01", "valor": valores[ano]} for ano in sorted(valores)],
    }

DESTINO.mkdir(parents=True, exist_ok=True)
for id_serie, serie in series.items():
    arquivo = DESTINO / f"{id_serie}.json"
    with arquivo.open("w", encoding="utf-8") as saida:
        json.dump(serie, saida, ensure_ascii=False, indent=2)
        saida.write("\n")
    print(f"{len(serie['dados'])} pontos escritos em {relativo(arquivo)}")

11 pontos escritos em indicadores/educacao/ideb-anos-iniciais.json
11 pontos escritos em indicadores/educacao/ideb-anos-finais.json
11 pontos escritos em indicadores/educacao/ideb-ensino-medio.json


## Validação inline

A última célula falha se o contrato quebrar, se a paridade bienal sair do lugar, se algum
arquivo gravado divergir do objeto ou se os valores deixarem de bater com a fonte. As
âncoras de valor são a defesa contra revisão silenciosa: se o Inep republicar um ano, a
célula falha e a revisão é registrada em `metodologia.md` antes de mexer no número.

In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
UNIDADE_ESPERADA = {"tipo": "numero", "simbolo": "índice", "casas": 1}

# Âncoras de valor por série. Falham numa revisão de fonte: confira a planilha da edição e
# registre a revisão em metodologia.md antes de mexer no número.
ANCORAS = {
    "ideb-anos-iniciais": {
        2005: 3.8, 2007: 4.2, 2009: 4.6, 2011: 5.0, 2013: 5.2, 2015: 5.5,
        2017: 5.8, 2019: 5.9, 2021: 5.8, 2023: 6.0, 2025: 6.3,
    },
    "ideb-anos-finais": {
        2005: 3.5, 2007: 3.8, 2009: 4.0, 2011: 4.1, 2013: 4.2, 2015: 4.5,
        2017: 4.7, 2019: 4.9, 2021: 5.1, 2023: 5.0, 2025: 5.3,
    },
    "ideb-ensino-medio": {
        2005: 3.4, 2007: 3.5, 2009: 3.6, 2011: 3.7, 2013: 3.7, 2015: 3.7,
        2017: 3.8, 2019: 4.2, 2021: 4.2, 2023: 4.3, 2025: 4.5,
    },
}

for id_serie, serie in series.items():
    assert set(serie) == CHAVES_RAIZ, (id_serie, sorted(set(serie) ^ CHAVES_RAIZ))
    assert serie["versao"] == 1, (id_serie, serie["versao"])
    assert serie["id"] == id_serie, (id_serie, serie["id"])
    assert serie["unidade"] == UNIDADE_ESPERADA, (id_serie, serie["unidade"])
    assert serie["eixo"] == "contagem", (id_serie, serie["eixo"])
    assert serie["medida"] == "nivel", (id_serie, serie["medida"])
    assert serie["granularidade"] == "bienal", (id_serie, serie["granularidade"])
    assert isinstance(serie["fonte"], str) and serie["fonte"], id_serie
    assert serie["urlFonte"].startswith("https://"), id_serie

    dados = serie["dados"]
    assert dados, f"{id_serie}: série vazia"
    datas = [ponto["data"] for ponto in dados]
    anos_lista = [int(data[:4]) for data in datas]

    # C15 bienal: todo ponto em AAAA-01-01, ordem ascendente, um por ano.
    assert datas[0] == f"{PRIMEIRO_ANO}-01-01", (id_serie, datas[0])
    assert datas[-1] == f"{EDICAO}-01-01", (id_serie, datas[-1])
    assert all(data.endswith("-01-01") for data in datas), f"{id_serie}: dia diferente de 01"
    assert all(a < b for a, b in zip(datas, datas[1:])), f"{id_serie}: datas fora de ordem"
    assert len(set(anos_lista)) == len(anos_lista), f"{id_serie}: ano repetido"

    # Paridade bienal ímpar: nenhum ano par, e nenhum ano ímpar faltando entre 2005 e a edição.
    assert all(ano % 2 == 1 for ano in anos_lista), (id_serie, anos_lista)
    assert anos_lista == list(range(PRIMEIRO_ANO, EDICAO + 1, 2)), (id_serie, anos_lista)
    assert len(dados) == len(anos_lista), (id_serie, len(dados))

    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, (id_serie, sorted(ponto))
        assert not isinstance(ponto["valor"], bool), (id_serie, ponto)
        assert isinstance(ponto["valor"], (int, float)), (id_serie, ponto)
        assert math.isfinite(ponto["valor"]), (id_serie, ponto)

    with (DESTINO / f"{id_serie}.json").open(encoding="utf-8") as aberto:
        assert json.load(aberto) == serie, f"{id_serie}: arquivo gravado difere do objeto"

    valores = {int(ponto["data"][:4]): ponto["valor"] for ponto in dados}
    assert valores == ideb[id_serie], f"{id_serie}: valor publicado difere da planilha"
    for ano, esperado in ANCORAS[id_serie].items():
        assert abs(valores[ano] - esperado) < 1e-9, (id_serie, ano, valores[ano], esperado)

    # Nas três etapas, 2025 é o maior valor da série iniciada em 2005.
    assert max(valores.values()) == valores[2025], f"{id_serie}: 2025 não é o máximo"
    print(f"ok: {id_serie} — {len(dados)} pontos ({datas[0]} a {datas[-1]})")

print("\nparidade ímpar, C15, ordem, contrato e valores conferidos contra a planilha do Inep.")

ok: ideb-anos-iniciais — 11 pontos (2005-01-01 a 2025-01-01)
ok: ideb-anos-finais — 11 pontos (2005-01-01 a 2025-01-01)
ok: ideb-ensino-medio — 11 pontos (2005-01-01 a 2025-01-01)

paridade ímpar, C15, ordem, contrato e valores conferidos contra a planilha do Inep.
